In [ ]:
from scipy.io import loadmat
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as  pd
from copy import deepcopy

from adaptive_latents import ArrayWithTime, proSVD

%matplotlib inline


In [ ]:
f = h5py.File('/mnt/data/datasets/daie21/Daie_et_al_2020_targeted_photostim.mat')



In [ ]:
n_sessions = f['data']['dt_si'].shape[0]
session_n = 0
dt = f[f['data']['dt_si'][session_n,0]][0,0]

rows = []

for l_or_r in 'LR':
    n_stim_groups = f[f['data'][l_or_r][session_n,0]].shape[0]
    for stim_group_n in range(n_stim_groups):
        data = f[f[f['data'][l_or_r][session_n,0]][stim_group_n,0]][:]

        if stim_group_n == 0:
            stim_start, stim_end = np.nan, np.nan
        else:
            stim_start, stim_end = f[f[f['data']['epochs'][session_n,0]]['stim'][stim_group_n-1,0]][:,0]

        accuracies = f[f[f['data']['C'+l_or_r][session_n,0]][stim_group_n,0]][:,0]
        for trial, accuracy in zip(data,accuracies):
            rows.append(dict(l_or_r=l_or_r, stim_group_n=stim_group_n, trial=trial.T, stim_start=stim_start, stim_end=stim_end, accuracy=accuracy))
_, n_neurons, n_timepoints = data.shape
t = np.arange(n_timepoints) * dt

df = pd.DataFrame(rows)
nan_rows = df[df['stim_start'].isna()].sample(frac=1).reset_index(drop=True)
non_nan_rows = df[df['stim_start'].notna()].sample(frac=1).reset_index(drop=True)
df = pd.concat([nan_rows, non_nan_rows], ignore_index=True)

In [ ]:
def concat_rows(df):
    trials = []
    stims = []
    row_count = 0
    for idx, row in df.iterrows():
        trial = row['trial']
        if np.var(trial) < .1:
            continue
        stim_start = row['stim_start']
        if not np.isnan(stim_start):
            group_vec = np.zeros(n_stim_groups-1)
            group_vec[row['stim_group_n']-1] = 1
            stims.append(ArrayWithTime(group_vec, (stim_start//dt + 1) * dt + row_count * dt))

        trials.append(trial)
        row_count += trial.shape[0]
        trials.append(np.empty([5,n_neurons]) * np.nan)
        row_count += trials[-1].shape[0]


    A = np.vstack(trials)
    A = ArrayWithTime(A, np.arange(A.shape[0]) * dt)
    stims = ArrayWithTime.from_list(stims)
    return A, stims
A, stims = concat_rows(df)
stims.t = stims.t- dt/50
# print(A.shape)
# print(stims.t)

In [ ]:
pro = proSVD(k=2)
latents = pro.offline_run_on(A)

In [ ]:
fig, ax = plt.subplots()

s = stims.t[5]
s = slice(s-2, s+2)
l = latents.slice_by_time(s)
ax.plot(l.t, l);

for t in stims.slice_by_time(s).t:
    ax.axvline(t, color='red', linestyle='--')
    ax.axvline(t+6*dt, color='k', linestyle='-')

In [ ]:
from adaptive_latents.stim_regressor import StimRegressor, StimAutoReg, StreamingKalmanFilter

sr1 = StimRegressor(error_on_missed_stim=False, stim_delay=2*dt, log_level=2, heed_stimuli=False, attempt_correction=False)

sr2 = StimRegressor(error_on_missed_stim=False, stim_delay=2*dt, log_level=2, heed_stimuli=True, attempt_correction=True)
sr2.stim_autoreg = StimAutoReg(n_steps_to_consider=7)
sr2.stim_reg.length_scales = np.array([2.98538262e-01, 8.91250938e-01, 2.37137371e-07])
sr2.stim_reg.reweight_every = np.inf


sr1.offline_run_on([(latents,'X'),(stims,'stim')])

new_stims = deepcopy(stims)
new_stims.t = new_stims.t - 2*dt
sr2.offline_run_on([(latents,'X'),(new_stims,'stim')])

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

pred_error = ArrayWithTime.from_list(sr1.log['pred_error'], squeeze_type='to_2d')
ax.plot(pred_error.t, np.linalg.norm(pred_error, axis=1))
print(np.nanmean(pred_error**2))

pred_error = ArrayWithTime.from_list(sr2.log['pred_error'], squeeze_type='to_2d')
ax.plot(pred_error.t, np.linalg.norm(pred_error, axis=1))
print(np.nanmean(pred_error**2))


ax.set_xlim([1230, 1250])

for t, stim in zip(sr2.stim_reg.input_histories[2], sr2.stim_reg.input_histories[1]):
    ax.axvline(t, color='red', linestyle='--')

In [ ]:
%matplotlib inline
from adaptive_latents.plotting_functions import AnimationManager, plot_history_with_tail


with AnimationManager('new_dataset_animation', outdir='.') as am:
    for t in np.linspace(1230, 1250, 20*20):
        ax = am.axs[0,0]
        ax.cla()

        plot_history_with_tail(ax, data=latents, current_t=t, tail_length=2, scatter_alpha=.9)
        ax.set_title(f't={t:.2f}')

        am.grab_frame()

In [ ]:
plt.plot(latents.t, latents)